# 02. Limpieza y preparación de datos

## Objetivo

En esta fase se preparará el dataset MxMH para el análisis estadístico, utilizando los problemas y características identificados durante la exploración inicial.

Se revisarán los valores ausentes y anómalos, se prepararán las variables categóricas y ordinales y se comprobará la consistencia de los datos resultantes.

Todas las transformaciones se documentarán y justificarán, procurando conservar la mayor cantidad posible de información relevante para las cuatro hipótesis del proyecto.

In [2]:
import pandas as pd
import numpy as np


## 1- crear copia

In [3]:
df = pd.read_csv("../data/mxmh_survey_results.csv")
# Copia independiente para trabajar
df_1 = df.copy()

print("Dataset original:", df.shape)
print("Dataset de trabajo:", df_1.shape)

Dataset original: (736, 33)
Dataset de trabajo: (736, 33)


## 2- Tratamiento de valores nulos 

Posibilidades:
A: Eliminar las filas: perdería 107 participantes, aproximadamente el 14,54 % de la muestra.

B: Imputar valores: podría sustituir los nulos por la mediana, pero estaría inventando valores de BPM que los participantes no proporcionaron.

C: Conservar los nulos: mantenego los participantes y dejo constancia de que desconozco sus BPM.

Opto por borrar la variable entera porque ninguna de las hipótesis necesita la variable BPM

In [ ]:
# Eliminar BPM de la copia de trabajo
df_1 = df_1.drop(columns=["BPM"])

print("Dataset de trabajo:", df_1.shape)

Dataset de trabajo: (736, 32)


### Tratamiento de la variable BPM

Durante la exploración inicial se identificaron 107 valores nulos en la variable `BPM` (14,54 % de los registros), además de dos valores anómalos de 624 y 999.999.999 BPM.

Se ha decidido excluir esta variable del dataset de trabajo porque no interviene directamente en ninguna de las cuatro hipótesis planteadas.

Esta decisión permite evitar imputaciones innecesarias y conservar los 736 participantes originales.

La columna permanece disponible en el dataset original, garantizando la trazabilidad de la transformación.

### 2.2 Revisar los valores nulos de Music effects (Relevante en Hipotesis 4)

In [7]:
df_1.loc[
    df_1["Music effects"].isna(),
    ["Age", "Hours per day", "Fav genre",
     "Anxiety", "Depression", "Insomnia", "OCD", "Music effects"]
]

,Age,Hours per day,Fav genre,Anxiety,Depression,Insomnia,OCD,Music effects
0,18.0,3.0,Latin,3.0,0.0,1.0,0.0,NaN
1,63.0,1.5,Rock,7.0,2.0,2.0,1.0,NaN
161,18.0,2.0,K pop,0.0,2.0,0.0,0.0,NaN
168,19.0,3.0,K pop,8.0,9.0,6.0,0.0,NaN
338,21.0,2.0,EDM,10.0,10.0,5.0,7.0,NaN
441,21.0,1.5,Folk,3.0,3.0,1.0,0.0,NaN
532,67.0,1.0,Rock,3.0,2.0,2.0,2.0,NaN
561,21.0,1.0,K pop,4.0,5.0,2.0,5.0,NaN


### Tratamiento de valores nulos en Music effects

La variable `Music effects` presenta 8 valores nulos (1,09 % del total).

Se han revisado los registros afectados y se ha comprobado que conservan información válida en las variables psicológicas y de hábitos musicales.

Se ha decidido mantener estos registros en el dataset de trabajo, sin imputar respuestas, ya que no existe información suficiente para determinar cómo perciben los efectos de la música.

Para contrastar la hipótesis H4 se utilizarán únicamente los 728 participantes con respuestas válidas en `Music effects`. En las demás hipótesis se conservarán estos ocho participantes siempre que dispongan de las variables necesarias.

Esta decisión permite preservar información y evitar la eliminación innecesaria de registros.

### 2.3 valores nulos restantes

In [8]:
df_1.isna().sum().loc[lambda x: x > 0].sort_values(ascending=False)

Music effects                8
Instrumentalist              4
Foreign languages            4
While working                3
Age                          1
Primary streaming service    1
Composer                     1
dtype: int64

### Tratamiento de los valores nulos restantes

Tras excluir la variable `BPM`, se han identificado valores nulos en siete columnas del dataset.

Las variables `Instrumentalist`, `Foreign languages`, `While working`, `Primary streaming service`, `Composer` y `Age` presentan entre uno y cuatro valores nulos cada una.

Se ha decidido conservar estos registros sin realizar imputaciones, dado que las variables afectadas no intervienen directamente en las cuatro hipótesis principales del proyecto.

Esta estrategia permite conservar los 736 participantes y evitar modificaciones innecesarias de los datos originales.

-Comprobación del dataset

In [9]:
print("Filas originales:", len(df))
print("Filas después de la preparación:", len(df_1))
print("Columnas actuales:", df_1.shape[1])

Filas originales: 736
Filas después de la preparación: 736
Columnas actuales: 32


## 3. Preparación de variables relevantes para el análisis

- Hipotesis 3: ¿Existe una relación entre la frecuencia de escucha de determinados géneros musicales y los indicadores de salud mental?

-  La columna `Frecuency` al contener variables ordinales el paso lógico es convertirlas a numeros para poder comparar rangos

In [10]:

mapa_frecuencia = {
    "Never": 0,
    "Rarely": 1,
    "Sometimes": 2,
    "Very frequently": 3
}

print(mapa_frecuencia)

{'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Very frequently': 3}


### 3.1 Sustitución y guardado en una nueva columna para poder comprobar las transformaciones

In [11]:
df_1["Frequency [Rock]_num"] = df_1["Frequency [Rock]"].map(mapa_frecuencia)

df_1[["Frequency [Rock]", "Frequency [Rock]_num"]].head(10)

,Frequency [Rock],Frequency [Rock]_num
0,Never,0
1,Very frequently,3
2,Rarely,1
3,Never,0
4,Never,0
5,Very frequently,3
6,Never,0
7,Never,0
8,Very frequently,3
9,Sometimes,2


In [12]:

columnas_frecuencia = [
    col for col in df_1.columns
    if col.startswith("Frequency [")
]

for col in columnas_frecuencia:
    df_1[col] = df_1[col].map(mapa_frecuencia)

In [13]:
df_1[columnas_frecuencia].head()

,Frequency [Classical],Frequency [Country],Frequency [EDM],Frequency [Folk],Frequency [Gospel],Frequency [Hip hop],Frequency [Jazz],Frequency [K pop],Frequency [Latin],Frequency [Lofi],Frequency [Metal],Frequency [Pop],Frequency [R&B],Frequency [Rap],Frequency [Rock],Frequency [Video game music],Frequency [Rock]_num
0,1,0,1,0,0,2,0,3,3,1,0,3,2,3,0,2,NaN
1,2,0,0,1,2,1,3,1,2,1,0,2,2,1,3,1,NaN
2,0,0,3,0,0,1,1,3,0,2,2,1,0,1,1,3,NaN
3,2,0,0,1,2,0,3,2,3,2,0,2,2,0,0,0,NaN
4,0,0,1,0,1,3,0,3,2,2,0,2,3,3,0,1,NaN


In [ ]:
df_1 = df_1.drop(columns=["Frequency [Rock]_num"])


In [24]:
columnas_frecuencia = [
    col for col in df_1.columns
    if col.startswith("Frequency [")
]

print("Número de columnas:", len(columnas_frecuencia))

df_1[columnas_frecuencia].head()

Número de columnas: 16


,Frequency [Classical],Frequency [Country],Frequency [EDM],Frequency [Folk],Frequency [Gospel],Frequency [Hip hop],Frequency [Jazz],Frequency [K pop],Frequency [Latin],Frequency [Lofi],Frequency [Metal],Frequency [Pop],Frequency [R&B],Frequency [Rap],Frequency [Rock],Frequency [Video game music]
0,1,0,1,0,0,2,0,3,3,1,0,3,2,3,0,2
1,2,0,0,1,2,1,3,1,2,1,0,2,2,1,3,1
2,0,0,3,0,0,1,1,3,0,2,2,1,0,1,1,3
3,2,0,0,1,2,0,3,2,3,2,0,2,2,0,0,0
4,0,0,1,0,1,3,0,3,2,2,0,2,3,3,0,1


### Comprobar valores nulos después de la codificación

In [ ]:

print("Valores nulos:", df_1[columnas_frecuencia].isna().sum().sum())


for col in columnas_frecuencia:
    print(col, ":", sorted(df_1[col].dropna().unique()))

Valores nulos: 0
Frequency [Classical] : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Frequency [Country] : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Frequency [EDM] : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Frequency [Folk] : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Frequency [Gospel] : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Frequency [Hip hop] : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Frequency [Jazz] : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Frequency [K pop] : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Frequency [Latin] : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Frequency [Lofi] : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Frequency [Metal] : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Frequency [Pop] : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Frequency [R&B] : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Frequency [Rap] : [np.int64(0), np

### Codificación de las variables ordinales

Se han transformado las 16 variables de frecuencia de escucha musical en valores numéricos mediante la siguiente correspondencia:

- `Never` → 0
- `Rarely` → 1
- `Sometimes` → 2
- `Very frequently` → 3

La transformación se ha aplicado mediante un diccionario y un bucle `for`, manteniendo el orden natural de las categorías.

Esta preparación permitirá utilizar las variables en el análisis de correlaciones de Spearman correspondiente a la hipótesis H3.

Tras la transformación, se ha verificado que las 16 columnas contienen valores entre 0 y 3 y que no se han generado valores ausentes adicionales.

In [26]:
df_1[columnas_frecuencia].dtypes

Frequency [Classical]           int64
Frequency [Country]             int64
Frequency [EDM]                 int64
Frequency [Folk]                int64
Frequency [Gospel]              int64
Frequency [Hip hop]             int64
Frequency [Jazz]                int64
Frequency [K pop]               int64
Frequency [Latin]               int64
Frequency [Lofi]                int64
Frequency [Metal]               int64
Frequency [Pop]                 int64
Frequency [R&B]                 int64
Frequency [Rap]                 int64
Frequency [Rock]                int64
Frequency [Video game music]    int64
dtype: object

## 4. Validación final del dataset preparado

- comprobaciones

In [27]:
print("Dimensiones originales:", df.shape)
print("Dimensiones actuales:", df_1.shape)

print("\nValores nulos originales:")
print(df.isna().sum().sum())

print("\nValores nulos actuales:")
print(df_1.isna().sum().sum())

print("\nValores nulos en indicadores psicológicos:")
indicadores = ["Anxiety", "Depression", "Insomnia", "OCD"]
print(df_1[indicadores].isna().sum())

Dimensiones originales: (736, 33)
Dimensiones actuales: (736, 32)

Valores nulos originales:
129

Valores nulos actuales:
22

Valores nulos en indicadores psicológicos:
Anxiety       0
Depression    0
Insomnia      0
OCD           0
dtype: int64


- Guardo resultado en nuevo Dataset

In [28]:
df_1.to_csv("../data/mxmh_preparado02.csv", index=False)

In [29]:
df_comprobacion = pd.read_csv("../data/mxmh_preparado02.csv")

print("Dimensiones:", df_comprobacion.shape)
print("Nulos:", df_comprobacion.isna().sum().sum())
print("Tipo de Frequency [Rock]:", df_comprobacion["Frequency [Rock]"].dtype)

Dimensiones: (736, 32)
Nulos: 22
Tipo de Frequency [Rock]: int64


## Resultado de la preparación de datos

Tras la limpieza y preparación, se ha obtenido un dataset de 736 registros y 32 columnas.

Se ha excluido la variable `BPM` por no ser necesaria para las hipótesis del proyecto y se han codificado numéricamente las 16 variables ordinales de frecuencia musical.

Se han conservado los valores ausentes restantes, evitando imputaciones y eliminaciones innecesarias de participantes.

El dataset preparado se ha exportado como `mxmh_preparado.csv`, que servirá como punto de partida para el análisis estadístico de las cuatro hipótesis.

El archivo original permanece sin modificaciones, lo que permite reproducir y verificar el proceso de preparación.